# M49 · detection 2D trên Colab qua Google Drive
Trong ứng dụng, nhập BDD100K/KITTI 2D và tạo lượt Colab. Mount Drive, điền mã lượt vào ROOT rồi chạy các cell.
Notebook kiểm tra checksum ảnh và tự ghi dự đoán/thời gian/model vào Drive. Backend tự nhận, không tải JSON bằng tay.
YOLO11s dùng lớp COCO: mặc định thử car/Car. Các lớp không tương ứng (rider, Cyclist, traffic sign…) cần checkpoint phù hợp, không đổi tên để giả lập.
Pipeline cần kiểm chứng trên Colab/GPU trước khi thu số liệu chính thức.
Tài liệu: https://docs.ultralytics.com/models/yolo11/


In [ ]:
%pip install ultralytics==8.3.203


In [ ]:
import json, time, hashlib, subprocess, sys
from pathlib import Path
import torch, ultralytics
from PIL import Image
from ultralytics import YOLO

# M49_DRIVE_TRANSPORT
"""Shared by generated notebook cells; stdlib only, testable without Colab."""
import hashlib
import json
from pathlib import Path


def load_drive_job(root, task):
    root = Path(root)
    request = json.loads((root/'request.json').read_text(encoding='utf-8'))
    if request['task'] != task or root.name != request['job_id']:
        raise ValueError('Sai notebook hoặc sai thư mục lượt chạy')
    names = request['manifest']
    if not isinstance(names, list) or not 1 <= len(names) <= 20 or len(names) != len(set(names)):
        raise ValueError('Manifest không hợp lệ')
    for name in names:
        if not isinstance(name, str) or not name or '/' in name or '\\' in name or ':' in name or name in {'.', '..'}:
            raise ValueError('Tên mẫu không hợp lệ')
        path = root/'assets'/name
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != request['sha256'].get(name):
            raise ValueError(f'{name}: chờ Drive đồng bộ đủ dữ liệu rồi chạy lại')
    return request


def publish_drive_result(root, request, bundle, timing, environment):
    root = Path(root)
    if root.name != request['job_id'] or bundle['task'] != request['task']:
        raise ValueError('Sai lượt chạy hoặc tác vụ')
    if sorted(s['asset_name'] for s in bundle['samples']) != sorted(request['manifest']):
        raise ValueError('Kết quả thiếu/trùng/thừa mẫu')
    if (root/'result.ready.json').exists():
        raise ValueError('Lượt này đã xuất kết quả. Tạo lượt mới trong ứng dụng để chạy lại.')
    raw = json.dumps(bundle, ensure_ascii=False, allow_nan=False, indent=2).encode('utf-8')
    if len(raw) > 32*1024*1024:
        raise ValueError('Kết quả vượt 32 MiB')
    outputs = {'predictions.json': raw,
               'timing.json': json.dumps(timing, ensure_ascii=False, allow_nan=False, indent=2).encode('utf-8'),
               'environment.txt': environment.encode('utf-8')}
    for name, data in outputs.items():
        temporary = root/(name+'.tmp')
        temporary.write_bytes(data)
        temporary.replace(root/name)
    receipt = {'job_id': request['job_id'], 'sha256': hashlib.sha256(raw).hexdigest()}
    temporary = root/'result.ready.tmp'
    temporary.write_text(json.dumps(receipt), encoding='utf-8')
    temporary.replace(root/'result.ready.json')
    return receipt

from google.colab import drive
drive.mount('/content/drive')
# Copy the job ID displayed in the app. This is the Colab path, not the Windows path.
ROOT = Path('/content/drive/MyDrive/M49/PASTE_JOB_ID')
request = load_drive_job(ROOT, 'detection_2d')

ASSETS = ROOT / 'assets'
# M49_MODEL_SELECTION
model_spec = request.get('model', {})
CHECKPOINT = model_spec.get('checkpoint', 'yolo11s.pt')
if model_spec.get('custom'):
    model_path = (ROOT.parent / CHECKPOINT).resolve()
    assert model_path.is_relative_to((ROOT.parent / 'models').resolve()), 'Checkpoint ngoài thư mục models'
    assert model_path.is_file(), 'Chờ Drive đồng bộ checkpoint'
    assert hashlib.sha256(model_path.read_bytes()).hexdigest() == model_spec['sha256'], 'Checkpoint chưa đồng bộ đủ'
    CHECKPOINT = str(model_path)

# Exact semantic aliases only; Cyclist/rider cannot be replaced by a person box.
aliases = {'pedestrian': 'person', 'Car': 'car', 'Pedestrian': 'person', 'Truck': 'truck', 'Tram': 'train'}
CLASS_MAP = {aliases.get(category, category): category for category in request['classes']}
assert len(CLASS_MAP) == len(request['classes']), 'Hai lớp dự án trùng một lớp model'

CONFIDENCE = model_spec.get('confidence', 0.25)  # Tune on pilot only, then freeze.
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def stamp():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.perf_counter()

names = request['manifest']
assert isinstance(names, list) and 1 <= len(names) <= 20
assert all(isinstance(n, str) and n and '/' not in n and chr(92) not in n and Path(n).name == n for n in names)
assert len(set(names)) == len(names), 'Tên ảnh trùng'
paths = [(ASSETS / name).resolve() for name in names]
assert all(p.parent == ASSETS.resolve() and p.is_file() for p in paths), 'Thiếu ảnh hoặc đường dẫn không hợp lệ'
assert CLASS_MAP and all(isinstance(v, str) and v.strip() for v in CLASS_MAP.values())
print('Số ảnh:', len(paths), 'Thiết bị:', DEVICE)


In [ ]:
start = stamp()
model = YOLO(CHECKPOINT)
load_seconds = stamp() - start  # Includes download if checkpoint is not cached.
assert model.task == 'detect', 'Cần checkpoint detection 2D'
CLASS_MAP = {(category if category in model.names.values() else aliases.get(category, category)): category for category in request['classes']}
assert set(CLASS_MAP) <= set(model.names.values()), 'Lớp cấu hình không có trong checkpoint'
class_ids = [i for i, name in model.names.items() if name in CLASS_MAP]
kwargs = dict(imgsz=IMGSZ, conf=CONFIDENCE, classes=class_ids, device=DEVICE,
              max_det=80, verbose=False, save=False)
start = stamp()
model.predict(str(paths[0]), **kwargs)
warmup_seconds = stamp() - start
samples, per_image = [], []
for name, path in zip(names, paths):
    with Image.open(path) as image:
        width, height = image.size
    start = stamp()
    result = model.predict(str(path), **kwargs)[0]
    predict_seconds = stamp() - start  # Image read, preprocess, forward, NMS.
    start = stamp()
    objects = []
    boxes = result.boxes
    for index, (xyxy, cls, conf) in enumerate(zip(boxes.xyxy.cpu().tolist(), boxes.cls.cpu().tolist(), boxes.conf.cpu().tolist())):
        x1, y1, x2, y2 = xyxy
        coords = [max(0., min(width, x1)), max(0., min(height, y1)),
                  max(0., min(width, x2)), max(0., min(height, y2))]
        assert coords[2] > coords[0] and coords[3] > coords[1], 'Box rỗng sau giới hạn biên'
        objects.append(dict(id=f'ai-{index}', category=CLASS_MAP[model.names[int(cls)]],
            confidence=float(conf), geometry=dict(kind='bbox2d', xyxy=coords)))
    samples.append(dict(asset_name=name, objects=objects, active_seconds=None))
    per_image.append(dict(asset_name=name, predict_pipeline_seconds=predict_seconds,
        export_conversion_seconds=stamp()-start, stages_ms=result.speed, reached_max_det=len(objects) >= 80))
print('Đã tạo nhãn cho', len(samples), 'ảnh. Nếu reached_max_det=true, kiểm tra khả năng bị cắt ở 80 box.')


In [ ]:
checkpoint_path = Path(model.ckpt_path)
checkpoint_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
bundle = dict(schema_version='1.0', task='detection_2d', name=(Path(CHECKPOINT).name + ' · 2D')[:120],
    run=dict(model=CHECKPOINT, checkpoint=f'{checkpoint_path.name} sha256:{checkpoint_hash}',
        gpu=gpu, library_version=f'ultralytics=={ultralytics.__version__}; torch=={torch.__version__}',
        inference_seconds=sum(r['predict_pipeline_seconds'] for r in per_image)), samples=samples)
timing = dict(load_and_download_seconds=load_seconds, warmup_seconds=warmup_seconds,
    per_image=per_image, training_performed=False, manifest=names,
    image_sha256={name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in zip(names, paths)},
    checkpoint_sha256=checkpoint_hash, class_map=CLASS_MAP, confidence=CONFIDENCE,
    imgsz=IMGSZ, max_det=80, device=gpu,
    timing_definition='predict_pipeline includes image read/preprocess/forward/NMS; export conversion separate. Human and upload times excluded.')
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
publish_drive_result(ROOT, request, bundle, timing, environment)
print('Đã ghi kết quả lên Drive. Backend đang chạy sẽ tự nhập sau khi Drive đồng bộ.')
